# pi 的对比对：采样、判分、组对（以及之后的生成把关）——Colab A100

融合 notebook（e16）的搜索和分量筛选不再用“关思考、数据集参考答案”的 NLL，改用 **pi 自己的推理里，对的比错的更受偏爱多少**。这个 notebook 准备这份数据：

1. **选题**：答案能自动判对错、考推理而不是考记忆、pi 大约能做对一半到八成的公开题，按固定 commit 下载：
   - 数学：Omni-MATH 里的数论题（难度 ≥ 4，答案是整数）；
   - 理科：JEE Advanced 的物理和化学题（JEEBench：单选、多选、整数、两位小数）；
   - 逻辑：MuSR（读一段故事做多步推理：推理破案、物品放置、分组安排）；
   - 中文：C-Eval 里官方标为“难”的科目中大学的 5 个（高等数学、离散数学、概率统计、大学物理、大学化学；高中的那几个多是记一个知识点）；
   - 代码：LiveCodeBench 的读代码求输出（中等和困难）。
   默认对比用 440 题，另留 130 题（完全不重叠）只用来测正确率。2026-11-05 之前用的 MATH 3–5 级、BBH、CRUXEval、C-Eval 理科、MMLU-Pro 对 pi 太容易（正确率 95%，600 题只凑出 19 对），已经换掉。
2. **采样**：pi 开思考，用它自己的 `generation_config` 采样参数，单条最长 8192 token。对比用的题每题 4 次，留出题每题 2 次。vLLM 跑在一张 A100 上，pi 以 FP8 加载（BF16 的 54GB 放不进 40GB）：这只影响“由谁来写这些推理”，融合 notebook 打分时用全精度。
3. **判分、组对**：只看 `</think>` 之后最后一个 `<answer>…</answer>`。同一道题里取一条写完且答对的，配一条失败的：
   - 先找写完但答错的，取长度最接近的一对（长度比不超过 2）；
   - 没有的话，用一条写到 token 上限被截断的，只取它和答对那条一样长的开头（难题上 pi 很少写完答错，多半是想太久没收敛）。开头里重复太多的不用：比同类答对回答的 95% 都重复（不重复 4-gram 的比例更低），就当作在打转，因为打转的文字很好预测，会让任何改动都显得更好；剩下的取最不重复的那条；
   - 答案解析不出的不进对比对。
4. **输出**：`pairs.jsonl`（每对两行，存 prompt 和回答的 token id，融合 notebook 逐 token 打分时和这里生成的完全一致）、`report.json`、pi 在留出题上的正确率（之后生成把关的基准）。

**生成把关（第 7 节，`STAGE = "veto"`）**：融合 notebook 选好模型后，会把候选（分量筛选的结果和全局搜索的结果）写到 Drive 并留一份 `veto/request.json`。到时把本 notebook 的 `STAGE` 改成 `"veto"` 再运行：在同一批留出题上开思考生成、判分，和 pi 比正确率；比 pi 低超过 5 个百分点的不通过。结果写回 `veto/result.json`，融合 notebook 读它决定上传哪一个。

**运行时**：Colab **A100**（40GB 即可，标准 RAM 就够）。采样约 2–4 小时（标准 RAM 的 A100 每小时约 6.7 个计算单元），每 128 道题存一次结果到 Drive，断开后从头运行全部单元格会接着做（按题号续跑）。本地盘要放 pi 的权重约 56GB。

**注意**：真实的 27B、vLLM、这台机器上的 FP8 都没有在这里运行过；选题、判分、组对和把关规则在 CPU 上用假数据测过（`tests/test_contrast_pairs.py`）。第一次运行如果某一步报错，把输出发回来。


## 1. 配置

一般只需确认 `STAGE`。

| 项 | 说明 |
|---|---|
| `STAGE` | `"pairs"`：选题、采样、判分、组对；`"regroup"`：样本都已经采完时，只按当前的规则重新判分、组对（不加载模型，CPU 运行时就行，几分钟）；`"veto"`：评融合 notebook 交来的候选 |
| `DRIVE_DIR` | 与融合 notebook 相同的记录目录；本 notebook 的结果在其下 `contrast/` |
| `SUBJECT_REPO` / `SUBJECT_REVISION` | pi。版本留空：用融合 notebook 锁定的 commit（`subject_pin.json`） |
| `COUNTS` | 每类的题数：`pool` 用来组对，`heldout` 只测正确率 |
| `SAMPLES_POOL` / `SAMPLES_HELDOUT` | 每题采样次数 |
| `MAX_RESPONSE_TOKENS` | 单条回答（思考 + 答案）最长多少 token，超出算没写完 |
| `MAX_PAIR_RATIO` | 一对里两条回答的长度比上限 |
| `VLLM` | vLLM 的设置：FP8 加载、显存占用比例、最多同时多少条 |
| `VLLM_VERSION` | 装哪个版本的 vLLM；默认锁定第一次跑通的 0.33.0，pi 和之后生成把关的候选用同一个版本 |
| `CHUNK` | 每批多少道题，每批结束存一次 |
| `VETO_MAX_DROP` | 生成把关：候选正确率比 pi 低多少就不通过（0.05 = 5 个百分点） |
| `AUTO_DISCONNECT` | `True`：全部跑完或中途出错后，先把 Drive 写完再自动断开运行时，停止计费（出错时报错另存到 `DRIVE_DIR/run_logs/`）。夜里无人看着时用 |


In [ ]:
import os, sys, json, shutil, subprocess, time, gc
from pathlib import Path

STAGE = "pairs"              # "pairs"：采样并组对；"regroup"：样本都已采完时只重新组对（CPU 运行时即可）；"veto"：评融合 notebook 交来的候选
DRIVE_DIR = "/content/drive/MyDrive/qwen38_efficientthink"
CONTRAST_DIR = DRIVE_DIR + "/contrast"
SUBJECT_REPO = "bytkim/Qwen3.8-27B-pi"
SUBJECT_REVISION = ""        # 留空：读融合 notebook 锁定的 DRIVE_DIR/subject_pin.json
LOCAL_MODEL = "/content/contrast_model"     # 本地盘：pi（或把关时的候选）的权重
COUNTS = {"math": {"pool": 130, "heldout": 40}, "science": {"pool": 110, "heldout": 30}, "logic": {"pool": 70, "heldout": 20}, "chinese_mc": {"pool": 70, "heldout": 20}, "code_reasoning": {"pool": 60, "heldout": 20}}
SAMPLES_POOL = 4
SAMPLES_HELDOUT = 2
MAX_RESPONSE_TOKENS = 8192
MAX_PAIR_RATIO = 2.0
THINKING = True
SEED = 20261104
VLLM = {"quantization": "fp8", "gpu_memory_utilization": 0.93, "max_num_seqs": 64}
CHUNK = 128                  # 每批题数：批越大，等最慢那条的空转越少；每批结束存一次
VETO_MAX_DROP = 0.05
INSTALL = True
VLLM_VERSION = "0.33.0"       # 第一次采样跑通的版本；pi 和候选（生成把关）要用同一个版本生成。留空则装最新版
AUTO_DISCONNECT = False      # True：全部跑完或出错后，自动断开运行时（停止计费）；夜里无人看着时用
NOTEBOOK_TAG = "contrast"


In [ ]:
# 夜里无人看着时用：AUTO_DISCONNECT = True 时，全部跑完或中途出错，都先把 Drive 写完，再断开运行时（停止计费）。
# 出错时报错另存一份到 DRIVE_DIR/run_logs/，断开后也能看。手动中断（停止按钮）不会触发断开。
import datetime as _dt
import traceback as _traceback
from pathlib import Path as _Path

RUN_LOG_DIR = _Path(DRIVE_DIR) / "run_logs"


def _release_runtime(reason):
    print(f"\n{reason}：先把 Drive 写完，再断开运行时（AUTO_DISCONNECT = True）。", flush=True)
    try:
        from google.colab import drive as _d
        _d.flush_and_unmount()
    except Exception as _e:                      # noqa: BLE001
        print("Drive 同步：", _e, flush=True)
    from google.colab import runtime as _rt
    _rt.unassign()


def _on_error(shell, etype, evalue, tb, tb_offset=None):
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    try:
        RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)
        _path = RUN_LOG_DIR / f"{NOTEBOOK_TAG}-{_dt.datetime.now():%Y%m%d-%H%M%S}-error.txt"
        _path.write_text("".join(_traceback.format_exception(etype, evalue, tb)), encoding="utf-8")
        print("报错已存到", _path, flush=True)
    except Exception as _e:                      # noqa: BLE001
        print("报错没能存下：", _e, flush=True)
    _release_runtime("出错了")


if AUTO_DISCONNECT:
    from IPython import get_ipython
    get_ipython().set_custom_exc((Exception,), _on_error)
    print("AUTO_DISCONNECT = True：全部跑完或出错后会自动断开运行时；出错时报错存在", RUN_LOG_DIR)
else:
    print("AUTO_DISCONNECT = False：跑完后运行时保持连接（继续计费），记得手动断开。")


## 2. 环境

挂载 Drive，检查 GPU（要 A100 一类、显存 ≥ 40GB 的卡），安装 vLLM。vLLM 会带它自己需要的 torch 版本：这一格在导入 torch 之前安装，所以不用重启运行时。


In [ ]:
from google.colab import drive as _drive
if not Path("/content/drive/MyDrive").exists():
    _drive.mount("/content/drive")
Path(CONTRAST_DIR).mkdir(parents=True, exist_ok=True)

NEED_GPU = STAGE != "regroup"            # regroup only reads the saved samples
if STAGE not in ("pairs", "regroup", "veto"):
    raise ValueError('STAGE 只能是 "pairs"、"regroup" 或 "veto"')
_smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if _smi.returncode == 0 and _smi.stdout.strip():
    GPU_NAME, _mem = [x.strip() for x in _smi.stdout.strip().splitlines()[0].split(",")]
    GPU_GIB = float(_mem) / 1024
else:
    GPU_NAME, GPU_GIB = None, 0.0
print(f"GPU：{GPU_NAME or '无'}，显存 {GPU_GIB:.1f} GiB；本地盘剩余 {shutil.disk_usage('/content').free / 1024**3:.0f} GiB")
if NEED_GPU and GPU_NAME is None:
    raise RuntimeError("没有 GPU：请在“运行时 → 更改运行时类型”里选 A100，然后从头运行。")
if NEED_GPU and GPU_GIB < 39:
    raise RuntimeError(f"显存只有 {GPU_GIB:.0f} GiB：27B 的 FP8 权重约 27GB，再加推理缓存，至少要 40GB 的 A100。请换 A100 运行时。")

def _pip(*args, timeout=1800):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0:
        raise RuntimeError("pip install " + " ".join(args) + " 失败：" + (r.stderr or r.stdout)[-800:])

if INSTALL and NEED_GPU:
    _t0 = time.time()
    print(f"安装 vLLM {VLLM_VERSION or '（最新版）'}：要下载约 3GB（含它自带的 torch），一般 5–15 分钟，过程中不显示进度……", flush=True)
    _pip(f"vllm=={VLLM_VERSION}" if VLLM_VERSION else "vllm")
    _pip("-U", "huggingface_hub", "pandas", "pyarrow")
    print(f"安装完成，用时 {(time.time() - _t0) / 60:.0f} 分钟", flush=True)

import importlib.metadata as _md
VERSIONS = {}
for _p in ("vllm", "torch", "transformers", "huggingface_hub"):
    try:
        VERSIONS[_p] = _md.version(_p)
    except _md.PackageNotFoundError:
        VERSIONS[_p] = None
print("依赖：", VERSIONS)
if NEED_GPU and not VERSIONS["vllm"]:
    raise RuntimeError("vLLM 没装上：把 INSTALL 设为 True 再运行这一格")
if not NEED_GPU:
    _pip("-U", "huggingface_hub", "pandas", "pyarrow")

TOK = os.environ.get("HF_TOKEN", "")
if not TOK:
    try:
        from google.colab import userdata
        TOK = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
if not TOK:
    from getpass import getpass
    TOK = getpass("Hugging Face token（要能读 pi 和公开评测集）: ")
TOK = TOK.strip()
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
api = HfApi(token=TOK or None)


## 3. 模块

选题、判分、组对、把关规则和 vLLM 调用都在下面这个内嵌模块里；单元测试直接从本 notebook 取出它的源码。


In [ ]:
import importlib.util
CONTRAST_SOURCE = r'''"""Contrast pairs from the subject's own reasoning, for the fusion notebook's search and component stage (e16).

The subject S (pi) answers answer-checkable problems with thinking on, several times each. Where it is sometimes right
and sometimes wrong, one right and one wrong trace of the same problem make a pair. A candidate F is then scored by how
much more it prefers the right trace over the wrong one than S does:

    gain(F) = mean over pairs of [l_F(wrong) - l_F(right)] - [l_S(wrong) - l_S(right)]

where l is the mean per-token NLL of the whole response (thinking and answer). Both traces are S's own writing, so
style and format cancel; what is left is which line of reasoning F finds more likely. A guard keeps l_F on the right
traces within a few percent of S (the fusion notebook enforces it).

Also here: a held-out set (disjoint problems) on which S's accuracy with thinking on is measured, and later the same
for the candidates the fusion notebook builds (the veto: a candidate more than `max_drop` below S is not uploaded).

Pure parts (problem parsing and selection, answer extraction and grading, pairing, the veto rule) need only the
standard library. The vLLM parts run on a GPU and are imported lazily.
"""
import ast
from fractions import Fraction
import hashlib
import json
import math
from pathlib import Path
import random
import re

VERSION = 'contrast-2026-11-05.4'
# What the samples depend on (it names the run's directory): grading changes do not discard samples, which are
# graded again from their text when they are read.
SAMPLES_VERSION = 'contrast-2026-11-05.2'

# problems per category: `pool` are sampled `samples_pool` times to find pairs; `heldout` are for accuracy only.
# 2026-11-05: problems that pi gets right about half to four fifths of the time, and that test reasoning rather than
# recall (the first run, on MATH levels 3-5, BBH, CRUXEval, C-Eval and MMLU-Pro, gave pi 95% and only 19 pairs).
COUNTS = {'math': {'pool': 130, 'heldout': 40}, 'science': {'pool': 110, 'heldout': 30},
          'logic': {'pool': 70, 'heldout': 20}, 'chinese_mc': {'pool': 70, 'heldout': 20},
          'code_reasoning': {'pool': 60, 'heldout': 20}}
# The college-level ones of C-Eval's own "hard" subjects (the high-school ones are mostly recall of one fact)
CEVAL_HARD = ('advanced_mathematics', 'discrete_mathematics', 'probability_and_statistics', 'college_chemistry',
              'college_physics')
MUSR_FILES = ('murder_mystery.csv', 'object_placements.csv', 'team_allocation.csv')
# Revisions left empty are pinned on first use (DRIVE_DIR/contrast/dataset_pins.json) and kept.
SOURCES = {
    'math': {'repo': 'KbsdJames/Omni-MATH', 'revision': '', 'files': ['test.jsonl'],
             'domains': ['Number Theory'], 'min_difficulty': 4.0},
    'science': {'repo': 'daman1209arora/jeebench', 'revision': '', 'files': ['test.json'], 'subjects': ['phy', 'chem']},
    'logic': {'repo': 'TAUR-Lab/MuSR', 'revision': '', 'files': list(MUSR_FILES)},
    'chinese_mc': {'repo': 'ceval/ceval-exam', 'revision': '617524a00b307ff6f9933702f724131fe12ca7ce',
                   'files': [f'{s}/{split}-00000-of-00001.parquet' for s in CEVAL_HARD for split in ('val', 'dev')]},
    'code_reasoning': {'repo': 'livecodebench/execution-v2', 'revision': '', 'files': ['data/test-00000-of-00001.parquet'],
                       'difficulties': ['medium', 'hard']},
}
GEN_EN = 'Think it through, then reply with only the final answer in the form <answer>...</answer> and nothing else.'
GEN_ZH = '请先思考，然后只输出最终答案，格式为 <answer>...</answer>，不要输出其他任何内容。'
GEN_KIND = {
    'number': ' The final answer must be a plain number: digits only, no units, commas or words.',
    'letter': ' The final answer is the letter of the correct option, for example <answer>C</answer>.',
    'literal': ' The final answer is the return value written as a Python literal, exactly as repr() would print it.',
    'word': ' The final answer is exactly the answer word or option label the question asks for.',
    'letter_zh': '最终答案是正确选项的字母，例如 <answer>B</answer>。',
    'letters': ' One or more options may be correct. The final answer is the letters of all correct options, for example <answer>ACD</answer>.',
    'numeric': ' The final answer must be a plain number correct to two decimal places: digits only, no units or words.',
}
LETTERS = 'ABCDEFGHIJ'
DEFAULT_SAMPLING = {'temperature': 0.6, 'top_p': 0.95, 'top_k': 20}


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=1) + '\n', encoding='utf-8')
    tmp.replace(path)


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows), encoding='utf-8')
    tmp.replace(path)


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]


# ---------------------------------------------------------------- problems
def last_boxed(text):
    start = text.rfind('\\boxed{')
    if start < 0:
        return None
    i, depth = start + len('\\boxed{'), 1
    for j in range(i, len(text)):
        depth += {'{': 1, '}': -1}.get(text[j], 0)
        if depth == 0:
            return text[i:j]
    return None


def _thousands(s):
    """'1,234' -> '1234'; any other comma (a list of answers such as '1, 2') is kept."""
    return s.replace(',', '') if re.fullmatch(r'-?\d{1,3}(,\d{3})+(\.\d+)?', s) else s


def _key(text):
    return hashlib.sha256(' '.join(str(text).split()).encode()).hexdigest()[:16]


def ceval_item(row, subject):
    letter = str(row['answer']).strip()
    options = [str(row.get(k, '')).strip() for k in 'ABCD']
    if letter not in 'ABCD' or len(letter) != 1 or not all(options):
        return None
    question = str(row['question']).strip() + '\n' + '\n'.join(f'{k}. {o}' for k, o in zip('ABCD', options))
    return {'key': f'{subject}/{row["id"]}', 'group': subject, 'question': question, 'lang': 'zh', 'kind': 'letter_zh',
            'expected': letter, 'options': 4}


def _plain_integer(answer):
    """An Omni-MATH answer that is a plain integer ('$12$', '\\boxed{1,024}', '-3'), as int; else None."""
    s = str(answer).strip().strip('$').strip()
    boxed = last_boxed(s)
    if boxed is not None:
        s = boxed
    s = _thousands(re.sub(r'\s|\\[!,;]|[{}]', '', s))
    return int(s) if re.fullmatch(r'-?\d+', s) else None


# "find the greatest GOOD number and the smallest BAD number": two answers asked for, one integer given
_TWO_TARGETS = re.compile(r'\b(greatest|largest|smallest|least|maximum|minimum)\b[^.?]{0,80}\band\s+(the\s+)?'
                          r'(greatest|largest|smallest|least|maximum|minimum)\b', re.I)


def omni_item(row, domains, min_difficulty):
    """An Omni-MATH problem in one of these domains, at least this difficulty, with a plain integer answer."""
    tags = row['domain'] if isinstance(row['domain'], list) else ast.literal_eval(str(row['domain']))
    hit = next((t for t in tags for d in domains if d in str(t)), None)
    try:
        difficulty = float(row['difficulty'])
    except (TypeError, ValueError):
        return None
    answer = _plain_integer(row['answer'])
    question = str(row['problem']).strip()
    if hit is None or difficulty < min_difficulty or answer is None or _TWO_TARGETS.search(question):
        return None
    group = ' -> '.join(str(hit).split(' -> ')[1:3]) or 'omni'
    return {'key': _key(question), 'group': group, 'level': difficulty, 'question': question, 'lang': 'en',
            'kind': 'number', 'expected': answer}


def jee_item(row, subjects):
    """A JEE Advanced physics / chemistry question (JEEBench): single-answer MCQ, multiple-answer MCQ, integer, or
    numeric (graded within 0.01, as JEEBench does). The options are part of the question text."""
    if row['subject'] not in subjects:
        return None
    gold, kind = str(row['gold']).strip(), str(row['type'])
    if kind == 'MCQ' and re.fullmatch(r'[A-D]', gold):
        item = {'kind': 'letter', 'expected': gold, 'options': 4}
    elif kind == 'MCQ(multiple)' and re.fullmatch(r'[A-D]+', gold):
        item = {'kind': 'letters', 'expected': ''.join(sorted(set(gold)))}
    elif kind == 'Integer' and re.fullmatch(r'-?\d+', gold):
        item = {'kind': 'number', 'expected': int(gold)}
    elif kind == 'Numeric' and re.fullmatch(r'-?\d+(\.\d+)?', gold):
        item = {'kind': 'numeric', 'expected': float(gold)}
    else:
        return None
    question = str(row['question']).strip()
    return {'key': f'{row["description"]}/{row["index"]}', 'group': f'{row["subject"]}/{kind}', 'question': question,
            'lang': 'en', **item}


def musr_item(row, name):
    """A MuSR problem (a long story, then a question with two to five options), as a lettered choice."""
    choices = ast.literal_eval(str(row['choices'])) if not isinstance(row['choices'], list) else row['choices']
    index = int(row['answer_index'])
    if not 2 <= len(choices) <= 5 or not 0 <= index < len(choices) or str(choices[index]) != str(row['answer_choice']):
        return None
    question = (str(row['narrative']).strip() + '\n\n' + str(row['question']).strip() + '\n'
                + '\n'.join(f'{LETTERS[k]}. {c}' for k, c in enumerate(choices)))
    return {'key': f'{name}/{_key(question)}', 'group': name.split('.')[0], 'question': question, 'lang': 'en',
            'kind': 'letter', 'expected': LETTERS[index], 'options': len(choices)}


def lcb_item(row, difficulties):
    """A LiveCodeBench execution problem (a contest solution and one call): what the call returns."""
    if str(row.get('difficulty')) not in difficulties:
        return None
    output, code, call = str(row['output']).strip(), str(row['code']).rstrip(), str(row['input']).strip()
    try:
        ast.literal_eval(output)
        ast.parse(code)
    except (ValueError, SyntaxError, MemoryError, RecursionError):
        return None
    question = f'```python\n{code}\n```\nWhat does `{call}` return?'
    return {'key': f'{row["question_id"]}/{row["id"]}', 'group': str(row['difficulty']), 'question': question, 'lang': 'en',
            'kind': 'literal', 'expected': output}


def candidates(category, fetch, reader, files=None):
    """All usable items of one category. fetch(repo, revision, filename) -> local path; reader(path) -> rows (jsonl,
    json, csv and parquet). files: unused (kept for the notebook's call)."""
    src = SOURCES[category]
    items = []
    for name in src['files']:
        group = name.split('/')[0]
        for index, row in enumerate(reader(fetch(src['repo'], src['revision'], name))):
            if category == 'math':
                item = omni_item(row, src['domains'], src['min_difficulty'])
            elif category == 'science':
                item = jee_item(row, src['subjects'])
            elif category == 'logic':
                item = musr_item(row, name)
            elif category == 'chinese_mc':
                item = ceval_item(row, group)
            elif category == 'code_reasoning':
                item = lcb_item(row, src['difficulties'])
            else:
                raise ValueError(f'不认识的类别：{category}')
            if item is not None:
                items.append({**item, 'source': f'{src["repo"]}@{src["revision"][:12]}:{name}#{item["key"]}'})
    return items


def prompt_text(item):
    if item['lang'] == 'zh':
        return f'{item["question"]}\n\n{GEN_ZH}{GEN_KIND[item["kind"]]}'
    return f'{item["question"]}\n\n{GEN_EN}{GEN_KIND[item["kind"]]}'


def _rank(seed, category, key):
    return hashlib.sha256(f'{seed}|{category}|{key}'.encode()).hexdigest()


def select(category, items, seed, pool, heldout, max_prompt_chars=12000):
    """Deduplicate, drop over-long prompts, interleave the groups in a seeded order; the first `pool` items are the pool,
    the next `heldout` the held-out set (disjoint by construction)."""
    seen, kept = set(), []
    for item in items:
        norm = ' '.join(item['question'].split())
        if norm in seen or len(prompt_text(item)) > max_prompt_chars:
            continue
        seen.add(norm)
        kept.append(item)
    groups = {}
    for item in sorted(kept, key=lambda x: _rank(seed, category, x['key'])):
        groups.setdefault(item['group'], []).append(item)
    order = sorted(groups, key=lambda g: _rank(seed, category, 'group:' + g))
    picked, need = [], pool + heldout
    while len(picked) < need and any(groups.values()):
        for g in order:
            if groups[g] and len(picked) < need:
                picked.append(groups[g].pop(0))
    if len(picked) < need:
        raise ValueError(f'{category}：只有 {len(picked)} 道合格题目，需要 {need} 道')
    out = []
    for n, item in enumerate(picked):
        split = 'pool' if n < pool else 'heldout'
        out.append({'pid': f'{category}-{_key(item["source"])[:10]}', 'split': split, 'category': category,
                    'prompt': prompt_text(item), **{k: v for k, v in item.items() if k != 'question'}})
    if len({p['pid'] for p in out}) != len(out):
        raise ValueError(f'{category}：题目编号重复')
    return out


def check_counts(counts):
    if set(counts) != set(COUNTS):
        raise ValueError(f'COUNTS 要给出这几类：{sorted(COUNTS)}')
    for c, v in counts.items():
        if set(v) != {'pool', 'heldout'} or any(type(x) is not int or x < 0 for x in v.values()) or v['pool'] < 1:
            raise ValueError(f'COUNTS["{c}"] 写成 {{"pool": 正整数, "heldout": 非负整数}}')
    return {c: dict(counts[c]) for c in COUNTS}


# ---------------------------------------------------------------- answers
_ANSWER = re.compile(r'<answer>(.*?)</answer>', re.S)


def extract_answer(text, thinking=True):
    """The content of the last <answer>...</answer> after the end of the thinking, or None (no closing </think> when
    thinking was on, or no answer tag)."""
    if thinking:
        if '</think>' not in text:
            return None
        text = text.rsplit('</think>', 1)[1]
    found = _ANSWER.findall(text)
    return found[-1].strip() if found else None


def _number(text):
    s = text.strip()
    boxed = last_boxed(s)
    if boxed is not None:
        s = boxed
    s = re.sub(r'\\text\{([^{}]*)\}', r'\1', s)
    s = _thousands(re.sub(r'[\s$]|\\[!,;]|\\\(|\\\)|[{}]', '', s).rstrip('.'))
    if not re.fullmatch(r'-?\d+(\.\d+)?', s):
        return None
    value = Fraction(s)                                  # exact: no float rounding of long integers or decimals
    return int(value) if value.denominator == 1 else value


def _letter(text, allowed):
    s = text.strip()
    m = re.match(r'^\(?([A-Z])\)?(?=$|[\s.:：、,，)])', s)
    if not m or m[1] not in allowed:
        return None
    return m[1]


def _literal(text):
    try:
        return ('value', ast.literal_eval(text.strip()))
    except (ValueError, SyntaxError, MemoryError, RecursionError, TypeError):
        return ('text', ' '.join(text.split()))


def grade(problem, pred):
    """True / False, or None when there is no parseable answer."""
    if pred is None:
        return None
    kind, expected = problem['kind'], problem['expected']
    if kind == 'number':
        value = _number(pred)
        return None if value is None else value == expected
    if kind in ('letter', 'letter_zh'):
        n = problem.get('options')                      # BBH option questions do not say how many (up to R)
        allowed = 'ABCDEFGHIJKLMNOPQR'[:int(n)] if n else 'ABCDEFGHIJKLMNOPQR'
        letter = _letter(pred, allowed)
        return None if letter is None else letter == expected
    if kind == 'literal':
        got, want = _literal(pred), _literal(expected)
        if want[0] == 'value' and got[0] != 'value':
            return None                                  # not a Python value at all: unparseable, never a "wrong" trace
        if got[0] == 'value' and want[0] == 'value':
            return got[1] == want[1] and type(got[1]) is type(want[1])
        return got[1] == want[1]
    if kind == 'word':
        return pred.strip().rstrip('.').strip().lower() == str(expected).lower()
    if kind == 'letters':
        s = re.sub(r'(?i)\b(and|or)\b|[和与及]', ' ', pred.strip())
        if not s or re.search(r'[^A-D\s,;()&、，]', s):
            return None
        got = set(re.findall(r'[A-D]', s))
        return None if not got else ''.join(sorted(got)) == expected
    if kind == 'numeric':
        value = _number(pred)
        # within 0.01 of the answer, as JEEBench grades; compared exactly (0.82 against 0.83 is within)
        return None if value is None else abs(Fraction(value) - Fraction(str(expected))) <= Fraction(1, 100)
    raise ValueError(f'不认识的答案类型：{kind}')


# ---------------------------------------------------------------- sampling bookkeeping
def chunks(problems, size):
    return [problems[i:i + size] for i in range(0, len(problems), size)]


def sample_seed(seed, pid, tag=''):
    return int(hashlib.sha256(f'{seed}|{tag}|{pid}'.encode()).hexdigest()[:8], 16)


def graded_samples(problem, outputs, thinking=True):
    """[{'k', 'text', 'response_ids', 'finish_reason', 'pred', 'correct'}] for one problem's raw outputs."""
    out = []
    for k, o in enumerate(outputs):
        finished = o['finish_reason'] == 'stop'
        pred = extract_answer(o['text'], thinking) if finished else None
        out.append({'k': k, 'text': o['text'], 'response_ids': [int(t) for t in o['token_ids']],
                    'finish_reason': o['finish_reason'], 'pred': pred, 'correct': grade(problem, pred)})
    return out


def regrade(problem, samples, thinking=True):
    """The samples of one problem with pred / correct worked out again from their text (grading rules may have changed
    since they were generated)."""
    out = []
    for s in samples:
        pred = extract_answer(s['text'], thinking) if s['finish_reason'] == 'stop' else None
        out.append({**s, 'pred': pred, 'correct': grade(problem, pred)})
    return out


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


TOKENIZER_FILES = ('tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja', 'special_tokens_map.json',
                   'added_tokens.json', 'vocab.json', 'merges.txt', 'generation_config.json')


def tokenizer_hashes(directory):
    """SHA-256 of the subject's tokenizer, chat-template and generation-config files present in `directory`."""
    return {n: sha256_file(Path(directory) / n) for n in TOKENIZER_FILES if (Path(directory) / n).is_file()}


def accuracy(problems, results):
    """Per category and overall: the mean over problems of the share of correct samples (unfinished or unparseable
    samples count as wrong), with per-problem values for paired comparisons."""
    per = {}
    for p in problems:
        samples = results[p['pid']]['samples']
        per[p['pid']] = (p['category'], sum(bool(s['correct']) for s in samples) / max(1, len(samples)))
    cats = sorted({c for c, _ in per.values()})
    by = {c: sum(v for cc, v in per.values() if cc == c) / sum(1 for cc, _ in per.values() if cc == c) for c in cats}
    return {'overall': sum(by.values()) / len(by) if by else 0.0, 'by_category': by,
            'overall_micro': sum(v for _, v in per.values()) / len(per) if per else 0.0,
            'rule': 'overall = mean of the category accuracies (each category weighs the same); overall_micro = mean over problems',
            'per_problem': {k: v for k, (_, v) in per.items()},
            'unfinished': sum(s['finish_reason'] != 'stop' for p in problems for s in results[p['pid']]['samples']),
            'unparseable': sum(s['finish_reason'] == 'stop' and s['correct'] is None
                               for p in problems for s in results[p['pid']]['samples'])}


# ---------------------------------------------------------------- pairs
def distinct4(ids):
    """Share of distinct 4-grams among all 4-grams of a token sequence (1 = no repetition; loops push it down)."""
    grams = [tuple(ids[i:i + 4]) for i in range(len(ids) - 3)]
    return len(set(grams)) / max(1, len(grams))


def _percentile(values, q):
    values = sorted(values)
    return values[min(len(values) - 1, max(0, int(q * (len(values) - 1))))] if values else None


def build_pairs(problems, results, prompt_ids, max_ratio=2.0, seed=0, use_cut=True, loop_quantile=0.05):
    """Rows for the fusion notebook: per pool problem one trace that finished with the right answer and one that failed.

    A failed trace is, first choice, one that finished with a wrong answer (the pair whose lengths are closest, ratio at
    most max_ratio; ties by a seeded hash). Without one, and with use_cut, a trace cut at the token limit: on hard
    problems pi rarely finishes wrong; it fails by not converging. Only the cut trace's first len(right) tokens are used,
    so the two are exactly as long; such a prefix is dropped when it repeats itself more than all but loop_quantile of
    the category's right traces do (distinct4), because a loop is cheap to predict and would reward any change. Among the
    rest, the least repetitive. Unparseable answers never enter a pair. Returns (rows, report)."""
    rows = []
    report = {'problems': 0, 'all_right': 0, 'all_wrong': 0, 'no_usable': 0, 'length_mismatch': 0, 'pairs': 0,
              'wrong_answer': 0, 'cut_prefix': 0, 'cut_too_repetitive': 0}
    by_category = {}
    pool = [p for p in problems if p['split'] == 'pool']
    floors = {}
    if use_cut:
        for cat in sorted({p['category'] for p in pool}):
            scores = [distinct4(s['response_ids']) for p in pool if p['category'] == cat
                      for s in results[p['pid']]['samples'] if s['finish_reason'] == 'stop' and s['correct']]
            floors[cat] = _percentile(scores, loop_quantile)
    report['repetition_floor'] = floors
    for p in pool:
        report['problems'] += 1
        samples = results[p['pid']]['samples']
        right = [s for s in samples if s['finish_reason'] == 'stop' and s['correct'] is True]
        wrong = [s for s in samples if s['finish_reason'] == 'stop' and s['correct'] is False]
        cut = [s for s in samples if s['finish_reason'] != 'stop'] if use_cut else []
        if not right:
            report['all_wrong' if wrong or cut else 'no_usable'] += 1
            continue
        if not wrong and not cut:
            report['all_right'] += 1
            continue
        best = None
        for r in right:
            for w in wrong:
                lr, lw = len(r['response_ids']), len(w['response_ids'])
                if not lr or not lw or max(lr, lw) / min(lr, lw) > max_ratio:
                    continue
                tie = hashlib.sha256(f'{seed}|{p["pid"]}|{r["k"]}|{w["k"]}'.encode()).hexdigest()
                score = (abs(math.log(lr / lw)), tie)
                if best is None or score < best[0]:
                    best = (score, r, w, list(w['response_ids']), 'wrong_answer')
        if best is None and cut:
            ordered = sorted(right, key=lambda s: (len(s['response_ids']), s['k']))
            r = ordered[len(ordered) // 2]                       # the right trace of middle length
            n = len(r['response_ids'])
            floor = floors.get(p['category'])
            options = []
            for c in cut:
                prefix = list(c['response_ids'][:n])
                if len(prefix) < n:
                    continue
                score = distinct4(prefix)
                if floor is not None and score < floor:
                    continue
                tie = hashlib.sha256(f'{seed}|{p["pid"]}|{r["k"]}|{c["k"]}'.encode()).hexdigest()
                options.append((-score, tie, prefix, c))
            if options:
                options.sort()
                best = (None, r, options[0][3], options[0][2], 'cut_prefix')
            else:
                report['cut_too_repetitive'] += 1
                continue
        if best is None:
            report['length_mismatch'] += 1
            continue
        _, r, failed, failed_ids, kind = best
        report['pairs'] += 1
        report[kind] += 1
        by_category[p['category']] = by_category.get(p['category'], 0) + 1
        for role, s, ids in (('correct', r, list(r['response_ids'])), ('wrong', failed, failed_ids)):
            rows.append({'id': f'{p["pid"]}/{role}', 'pair': p['pid'], 'role': role, 'category': p['category'],
                         'source': p['source'], 'messages': [{'role': 'user', 'content': p['prompt']}],
                         'prompt_ids': list(prompt_ids[p['pid']]), 'response_ids': ids,
                         'expected': p['expected'], 'pred': s['pred'], 'sample': s['k'],
                         'failure': None if role == 'correct' else kind})
    report['by_category'] = dict(sorted(by_category.items()))
    report['tokens'] = sum(len(r['prompt_ids']) + len(r['response_ids']) for r in rows)
    report['response_tokens'] = sum(len(r['response_ids']) for r in rows)
    return rows, report


def check_pairs(rows):
    """The fusion notebook's check of a pairs file: two rows per pair, one right and one wrong, same prompt."""
    pairs = {}
    for r in rows:
        for key in ('id', 'pair', 'role', 'category', 'prompt_ids', 'response_ids'):
            if key not in r:
                raise ValueError(f'对比对文件缺少字段 {key}')
        if r['role'] not in ('correct', 'wrong') or not r['response_ids'] or not r['prompt_ids']:
            raise ValueError(f'{r["id"]}：role 只能是 correct / wrong，且 prompt_ids、response_ids 非空')
        pairs.setdefault(r['pair'], {})
        if r['role'] in pairs[r['pair']]:
            raise ValueError(f'{r["pair"]}：同一个 role 出现两次')
        pairs[r['pair']][r['role']] = r
    for pid, two in pairs.items():
        if set(two) != {'correct', 'wrong'}:
            raise ValueError(f'{pid}：对比对不完整')
        if two['correct']['prompt_ids'] != two['wrong']['prompt_ids'] or two['correct']['category'] != two['wrong']['category']:
            raise ValueError(f'{pid}：两条回答的题目不同')
    if len({r['id'] for r in rows}) != len(rows):
        raise ValueError('对比对文件 id 重复')
    return pairs


# ---------------------------------------------------------------- veto
def paired_difference(per_a, per_b, repeats=2000, seed=0):
    """Mean of b - a over the problems both have, with a bootstrap 95% interval (problems resampled)."""
    keys = sorted(set(per_a) & set(per_b))
    if not keys:
        raise ValueError('两边没有共同的题')
    diffs = [per_b[k] - per_a[k] for k in keys]
    rng = random.Random(seed)
    boots = sorted(sum(rng.choice(diffs) for _ in diffs) / len(diffs) for _ in range(int(repeats)))
    return {'diff': sum(diffs) / len(diffs), 'ci95': [boots[int(0.025 * repeats)], boots[int(0.975 * repeats) - 1]],
            'problems': len(keys)}


def veto(subject, candidates, max_drop=0.05):
    """The fusion notebook's rule: candidates in order of preference (e.g. the component-stage choice, then the global
    search's F); the first whose overall accuracy is at most max_drop below S's is approved. Returns the decision."""
    rows = []
    for name, acc in candidates:
        cmp = paired_difference(subject['per_problem'], acc['per_problem'])
        ok = acc['overall'] >= subject['overall'] - float(max_drop)
        rows.append({'name': name, 'overall': acc['overall'], 'by_category': acc['by_category'], 'vs_subject': cmp, 'ok': ok})
    chosen = next((r['name'] for r in rows if r['ok']), None)
    return {'subject': {'overall': subject['overall'], 'by_category': subject['by_category']}, 'candidates': rows,
            'approved': chosen, 'max_drop': float(max_drop)}


# ---------------------------------------------------------------- GPU side (vLLM)
def sampling_params_from(generation_config):
    """temperature / top_p / top_k (and min_p, presence / repetition penalty when given) from the subject's own
    generation_config.json, with Qwen's thinking-mode defaults for what is missing."""
    out = dict(DEFAULT_SAMPLING)
    for key in ('temperature', 'top_p', 'top_k', 'min_p', 'presence_penalty', 'repetition_penalty'):
        if key in (generation_config or {}) and generation_config[key] is not None:
            out[key] = generation_config[key]
    return out


def chat_prompt_ids(tokenizer, prompt, thinking=True):
    text = tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True,
                                         enable_thinking=thinking)
    return [int(t) for t in tokenizer(text, add_special_tokens=False)['input_ids']]


def make_llm(model_dir, max_model_len, gpu_memory_utilization=0.9, quantization='fp8', max_num_seqs=64, seed=0):
    """vLLM on one GPU. FP8 weights (dynamic, weight-only on Ampere): a 27B model in 40 GB; text only (no memory for
    the vision encoder). A failed start is tried once more with the same settings (a cold first start, which reads
    and quantizes all the weights, can fail where the second, from the file cache, does not); only then, or when vLLM
    rejects an argument, without the text-only setting."""
    from vllm import LLM
    base = dict(model=str(model_dir), tokenizer=str(model_dir), dtype='bfloat16', quantization=quantization,
                max_model_len=int(max_model_len), gpu_memory_utilization=float(gpu_memory_utilization),
                max_num_seqs=int(max_num_seqs), seed=int(seed), trust_remote_code=False)
    text_only = {'limit_mm_per_prompt': {'image': 0, 'video': 0}}
    error = None
    plan = [text_only, text_only, {}, {}]
    n = 0
    while n < len(plan):
        extra = plan[n]
        try:
            return LLM(**base, **extra)
        except TypeError as exc:                  # an argument this vLLM does not take: go on without it
            error = exc
            print(f'vLLM 不接受 {sorted(extra)}（{exc}）', flush=True)
            if extra:
                n = plan.index({})
                continue
        except Exception as exc:                  # noqa: BLE001 - shown, then tried again
            error = exc
            print(f'vLLM 第 {n + 1} 次启动失败（{type(exc).__name__}: {str(exc)[:300]}）。真正的原因在 vLLM 子进程的日志里：'
                  '代码执行程序 → 查看运行时日志。' + ('用同样的设置再试一次。' if n % 2 == 0 else ''), flush=True)
        n += 1
        import gc
        gc.collect()
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:                         # noqa: BLE001
            pass
    raise error


def generate(llm, prompt_ids, n, params, max_tokens, seeds):
    """Raw outputs per prompt: [[{'text', 'token_ids', 'finish_reason'}] * n]."""
    from vllm import SamplingParams
    sps = [SamplingParams(n=int(n), max_tokens=int(max_tokens), seed=int(s), skip_special_tokens=False, **params)
           for s in seeds]
    outs = llm.generate([{'prompt_token_ids': list(ids)} for ids in prompt_ids], sps, use_tqdm=True)
    return [[{'text': o.text, 'token_ids': list(o.token_ids), 'finish_reason': o.finish_reason} for o in out.outputs]
            for out in outs]
'''
_mod_dir = Path('/content/contrast_modules'); _mod_dir.mkdir(parents=True, exist_ok=True)
(_mod_dir / 'contrast_pairs.py').write_text(CONTRAST_SOURCE, encoding='utf-8')
_spec = importlib.util.spec_from_file_location('contrast_pairs', _mod_dir / 'contrast_pairs.py')
contrast_pairs = importlib.util.module_from_spec(_spec); sys.modules['contrast_pairs'] = contrast_pairs
_spec.loader.exec_module(contrast_pairs)
cp = contrast_pairs
print('contrast_pairs', cp.VERSION)


## 4. 锁定版本、选题

pi 的 commit 用融合 notebook 锁定的那个；评测集按固定 commit 下载（MATH 全集第一次用时锁定，记在 `contrast/dataset_pins.json`）。题目和它们的 prompt token 存到 `contrast/<指纹>/problems.jsonl`，之后的会话直接沿用。


In [ ]:
COUNTS = cp.check_counts(COUNTS)
_pin_file = Path(DRIVE_DIR) / "subject_pin.json"
_pin = json.loads(_pin_file.read_text()) if _pin_file.exists() else None
if SUBJECT_REVISION:
    SUBJECT_PIN = {"repo": SUBJECT_REPO, "revision": SUBJECT_REVISION, "subfolder": ""}
elif _pin and _pin.get("repo") == SUBJECT_REPO:
    SUBJECT_PIN = _pin
else:
    # No shared pin yet: lock S here and write the pin the fusion notebook reads (same file, same format), so that both
    # notebooks use one commit of S.
    _info = api.model_info(SUBJECT_REPO)
    _names = [s.rfilename for s in _info.siblings]
    if "config.json" not in _names or not any(n.endswith(".safetensors") and "/" not in n for n in _names):
        raise RuntimeError("pi 的权重不在仓库根目录：先运行一次融合 notebook 第 4 节（它会写 DRIVE_DIR/subject_pin.json），再运行本 notebook")
    SUBJECT_PIN = {"repo": SUBJECT_REPO, "revision": _info.sha, "subfolder": ""}
    cp.atomic_json(_pin_file, SUBJECT_PIN)
    print("已锁定 pi 的 commit，写入", _pin_file, "（融合 notebook 读同一个文件）")
if SUBJECT_PIN.get("subfolder"):
    raise RuntimeError("pi 的权重在子目录里：这个 notebook 还不支持，请把输出发回来")
print(f"pi：{SUBJECT_PIN['repo']} @ {SUBJECT_PIN['revision'][:12]}")

# the tokenizer, chat template and generation config first (a few MB)
snapshot_download(SUBJECT_PIN["repo"], revision=SUBJECT_PIN["revision"], local_dir=LOCAL_MODEL, token=TOK or None,
                  allow_patterns=["*.json", "*.jinja", "*.txt", "tokenizer*", "*.model"])
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
_gc_path = Path(LOCAL_MODEL) / "generation_config.json"
SAMPLING = cp.sampling_params_from(json.loads(_gc_path.read_text()) if _gc_path.exists() else {})
print("采样参数（来自 pi 的 generation_config）：", SAMPLING)

# dataset pins
_pins_path = Path(CONTRAST_DIR) / "dataset_pins.json"
DATASET_PINS = json.loads(_pins_path.read_text()) if _pins_path.exists() else {}
for _cat, _src in cp.SOURCES.items():
    if not _src["revision"]:
        if _src["repo"] not in DATASET_PINS:
            DATASET_PINS[_src["repo"]] = api.dataset_info(_src["repo"]).sha
        _src["revision"] = DATASET_PINS[_src["repo"]]
cp.atomic_json(_pins_path, DATASET_PINS)

RUN_IDENTITY = {"version": cp.SAMPLES_VERSION, "subject": SUBJECT_PIN, "counts": COUNTS, "seed": SEED, "samples_pool": SAMPLES_POOL,
                "samples_heldout": SAMPLES_HELDOUT, "max_response_tokens": MAX_RESPONSE_TOKENS, "thinking": THINKING,
                "sampling": SAMPLING, "sources": cp.SOURCES, "max_pair_ratio": MAX_PAIR_RATIO}
WORK = Path(CONTRAST_DIR) / cp.digest(RUN_IDENTITY)[:16]
WORK.mkdir(parents=True, exist_ok=True)
cp.atomic_json(WORK / "identity.json", RUN_IDENTITY)

_problems_path = WORK / "problems.jsonl"
if _problems_path.exists():
    PROBLEMS = cp.read_jsonl(_problems_path)
    print("沿用已选好的题：", _problems_path)
else:
    import pandas as pd
    _data_dir = Path("/content/contrast_data")
    def _fetch(repo, revision, name):
        return hf_hub_download(repo, name, repo_type="dataset", revision=revision, local_dir=str(_data_dir / repo.replace("/", "__")),
                               token=TOK or None)
    def _reader(path):
        path = str(path)
        if path.endswith(".jsonl"):
            return cp.read_jsonl(path)
        if path.endswith(".json"):
            return json.loads(Path(path).read_text(encoding="utf-8"))
        if path.endswith(".csv"):
            return pd.read_csv(path).to_dict("records")
        return pd.read_parquet(path).to_dict("records")
    PROBLEMS = []
    for _cat in cp.SOURCES:
        _items = cp.candidates(_cat, _fetch, _reader)
        _picked = cp.select(_cat, _items, SEED, COUNTS[_cat]["pool"], COUNTS[_cat]["heldout"])
        print(f"{_cat}：{len(_items)} 道可用，选 {len(_picked)} 道")
        PROBLEMS += _picked
    for _p in PROBLEMS:
        _p["prompt_ids"] = cp.chat_prompt_ids(tokenizer, _p["prompt"], THINKING)
    cp.write_jsonl(_problems_path, PROBLEMS)
PROMPT_IDS = {p["pid"]: p["prompt_ids"] for p in PROBLEMS}
_lens = sorted(len(v) for v in PROMPT_IDS.values())
print(f"题目：对比用 {sum(p['split'] == 'pool' for p in PROBLEMS)} 道，留出 {sum(p['split'] == 'heldout' for p in PROBLEMS)} 道；"
      f"prompt 长度中位 {_lens[len(_lens) // 2]}、最长 {_lens[-1]} token")
print("示例 prompt：\n" + tokenizer.decode(PROMPT_IDS[PROBLEMS[0]["pid"]])[:1500])
MAX_MODEL_LEN = _lens[-1] + MAX_RESPONSE_TOKENS + 16


## 5. 加载 pi（vLLM，FP8）

下载 pi 的完整权重（约 56GB）到本地盘，用 vLLM 以 FP8 加载。`STAGE = "veto"` 时跳过这一节（把关时加载的是候选）。


In [ ]:
LLM = None
if STAGE == "pairs":
    snapshot_download(SUBJECT_PIN["repo"], revision=SUBJECT_PIN["revision"], local_dir=LOCAL_MODEL, token=TOK or None)
    LLM = cp.make_llm(LOCAL_MODEL, MAX_MODEL_LEN, VLLM["gpu_memory_utilization"], VLLM["quantization"], VLLM["max_num_seqs"], SEED)
    print("vLLM 已加载 pi；最长上下文", MAX_MODEL_LEN)


## 6. 采样、判分、组对

对比题每题 `SAMPLES_POOL` 次、留出题每题 `SAMPLES_HELDOUT` 次；每 `CHUNK` 道题存一次（`samples/`），断开后从头运行全部单元格会按题号跳过已完成的。每道题的随机种子固定，重跑结果相同。全部采完后判分、组对，写出 `pairs.jsonl`、`pairs.manifest.json` 和 `report.json`（报告里有 `wrong_answer`：配写完答错的对数，`cut_prefix`：配截断开头的对数，`cut_too_repetitive`：只有打转的截断回答、没组成对的题数）。

样本都已经采完、只是组对规则改了时，用 `STAGE = "regroup"`：不装 vLLM、不下载 pi，CPU 运行时几分钟就重新写出这三个文件。


In [ ]:
def load_samples(problems, split, n, out_dir):
    """{pid: row} of the problems of `split` whose n samples are already in out_dir (any batch file, whatever the batch
    size was), graded again from their text with the current rules."""
    by_pid = {p["pid"]: p for p in problems if p["split"] == split}
    found = {}
    for path in sorted(Path(out_dir).glob(f"{split}-*.jsonl")):
        for row in cp.read_jsonl(path):
            p = by_pid.get(row["pid"])
            if p is not None and len(row["samples"]) == n:
                found[row["pid"]] = {**row, "samples": cp.regrade(p, row["samples"], THINKING)}
    return found


def run_samples(llm, problems, split, n, out_dir, tag):
    """Sample every problem of one split; returns {pid: {'samples': [...]}}. Problems already sampled are reused by
    their id (changing CHUNK loses nothing); the rest go in batches of CHUNK, each written as soon as it is done."""
    todo = [p for p in problems if p["split"] == split]
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    results = load_samples(problems, split, n, out_dir)
    missing = [p for p in todo if p["pid"] not in results]
    if missing and llm is None:
        raise RuntimeError(f"{split}：还有 {len(missing)} 道题没有采样，但这一阶段没有加载模型")
    batches = cp.chunks(missing, CHUNK)
    if results:
        print(f"{split}：沿用已采样的 {len(results)} 道题，还要采 {len(missing)} 道", flush=True)
    started = time.perf_counter(); tokens = 0
    for i, batch in enumerate(batches):
        outs = cp.generate(llm, [PROMPT_IDS[p["pid"]] for p in batch], n, SAMPLING, MAX_RESPONSE_TOKENS,
                           [cp.sample_seed(SEED, p["pid"], tag) for p in batch])
        rows = []
        for p, o in zip(batch, outs):
            rows.append({"pid": p["pid"], "category": p["category"], "samples": cp.graded_samples(p, o, THINKING)})
            tokens += sum(len(x["token_ids"]) for x in o)
        cp.write_jsonl(out_dir / f"{split}-{cp.digest([p['pid'] for p in batch])[:16]}.jsonl", rows)
        for row in rows:
            results[row["pid"]] = row
        el = time.perf_counter() - started
        print(f"{split} 第 {i + 1}/{len(batches)} 批：本次已生成 {tokens / 1e6:.2f}M token，{tokens / el:.0f} token/s，"
              f"估计还需 {el / (i + 1) * (len(batches) - i - 1) / 60:.0f} 分钟", flush=True)
    return results

if STAGE in ("pairs", "regroup"):
    RESULTS = {}
    RESULTS.update(run_samples(LLM, PROBLEMS, "pool", SAMPLES_POOL, WORK / "samples", "pool"))
    RESULTS.update(run_samples(LLM, PROBLEMS, "heldout", SAMPLES_HELDOUT, WORK / "samples", "heldout"))
    POOL = [p for p in PROBLEMS if p["split"] == "pool"]
    HELDOUT = [p for p in PROBLEMS if p["split"] == "heldout"]
    ACC_POOL = cp.accuracy(POOL, RESULTS)
    ACC_HELDOUT = cp.accuracy(HELDOUT, RESULTS)
    ROWS, PAIR_REPORT = cp.build_pairs(PROBLEMS, RESULTS, PROMPT_IDS, MAX_PAIR_RATIO, SEED)
    cp.check_pairs(ROWS)
    cp.write_jsonl(WORK / "pairs.jsonl", ROWS)
    # Who wrote these traces, with which tokenizer: the fusion notebook checks this against its S before scoring.
    cp.atomic_json(WORK / "pairs.manifest.json", {
        "version": cp.VERSION, "subject": SUBJECT_PIN, "tokenizer": cp.tokenizer_hashes(LOCAL_MODEL), "thinking": THINKING,
        "run_identity": cp.digest(RUN_IDENTITY), "quantization": VLLM["quantization"], "pairs": PAIR_REPORT["pairs"],
        "pairs_sha256": cp.sha256_file(WORK / "pairs.jsonl")})
    cp.atomic_json(WORK / "heldout_subject.json", {"identity": RUN_IDENTITY, "accuracy": ACC_HELDOUT,
                                                   "quantization": VLLM["quantization"]})
    REPORT = {"identity": RUN_IDENTITY, "versions": VERSIONS, "gpu": GPU_NAME, "pairs": PAIR_REPORT,
              "accuracy_pool": {k: v for k, v in ACC_POOL.items() if k != "per_problem"},
              "accuracy_heldout": {k: v for k, v in ACC_HELDOUT.items() if k != "per_problem"}}
    cp.atomic_json(WORK / "report.json", REPORT)
    print(json.dumps({k: v for k, v in REPORT.items() if k not in ("identity",)}, ensure_ascii=False, indent=1))
    print(f"\n对比对：{PAIR_REPORT['pairs']} 对，回答共 {PAIR_REPORT['response_tokens'] / 1e6:.2f}M token。")
    print("在融合 notebook（e16）第 1 节填：")
    print(f'CONTRAST_PAIRS_FILE = "{WORK / "pairs.jsonl"}"')


## 7. 生成把关（`STAGE = "veto"`）

融合 notebook 选好模型后，会把候选写到 Drive 并留下 `DRIVE_DIR/veto/request.json`。把第 1 节的 `STAGE` 改成 `"veto"`，在 A100 上从头运行：候选按偏好顺序（先分量筛选的结果，再全局搜索的结果）一个一个评：复制到本地盘、用 vLLM（FP8，和 pi 相同）在同一批留出题上、用同样的种子各生成 `SAMPLES_HELDOUT` 次，判分，和 pi 比正确率。第一个“比 pi 低不超过 5 个百分点”的通过，后面的就不评了，结果写进 `veto/result.json`；融合 notebook 读它决定上传哪个。

**比的是什么**：`overall` 是五类正确率的等权平均（每类都只算五分之一，不管题数多少），5 个百分点的门槛作用在它上面；留出题的简单平均（`overall_micro`）、各类正确率和配对差的 bootstrap 区间都写进结果，但不参与通过与否。pi 和候选的成绩都在同一个评测身份下（同一批题、同样的采样设置、种子、token 上限、量化方式和判分规则）算出；身份变了，候选会重新生成，不会混用旧成绩。

**一次运行只加载一个候选**：vLLM 在同一个进程里先后加载两个 27B 模型时，显存不一定能释放干净。第一个候选没通过、还要评第二个时，本节会停下并提示：用“代码执行程序 → 重启会话”后再从头运行，已评完的候选（`veto/eval-<名字>.json`）和已生成的批次（`veto/samples/`）都会沿用。


In [ ]:
if STAGE == "veto":
    _req_path = Path(DRIVE_DIR) / "veto" / "request.json"
    if not _req_path.exists():
        raise RuntimeError(f"没有 {_req_path}：融合 notebook 还没有交来候选")
    REQUEST = json.loads(_req_path.read_text())
    _subject_path = WORK / "heldout_subject.json"
    if not _subject_path.exists():
        raise RuntimeError("还没有 pi 在留出题上的结果：先用 STAGE = \"pairs\" 跑完采样")
    _subject = json.loads(_subject_path.read_text())
    if _subject.get("quantization", VLLM["quantization"]) != VLLM["quantization"]:
        raise RuntimeError(f"pi 是按 {_subject['quantization']} 评的，现在设的是 {VLLM['quantization']}：两边要一样才能比")
    HELDOUT = [p for p in PROBLEMS if p["split"] == "heldout"]
    # pi's held-out samples of this run, graded with the current rules (the candidates' are graded the same way)
    SUBJECT_ACC = cp.accuracy(HELDOUT, run_samples(None, PROBLEMS, "heldout", SAMPLES_HELDOUT, WORK / "samples", "heldout"))
    # Everything the comparison depends on besides the candidate: the run (pi, problems, sampling, seeds, token limit),
    # the quantization and the grading code. A candidate's samples are kept and reused only under this identity.
    EVAL_ID = cp.digest({"run": cp.digest(RUN_IDENTITY), "quantization": VLLM["quantization"],
                         "heldout": sorted(p["pid"] for p in HELDOUT), "grading": cp.VERSION})
    EVALS = []
    DECISION = None
    _loaded = False
    for cand in REQUEST["candidates"]:              # in order of preference; the first that passes is taken
        out = _req_path.parent / f"eval-{cand['name']}.json"
        _dir = _req_path.parent / "samples" / EVAL_ID[:16] / f"{cand['name']}-{cand['fingerprint'][:16]}"
        _have = load_samples(PROBLEMS, "heldout", SAMPLES_HELDOUT, _dir)
        if len(_have) == len(HELDOUT):
            acc = cp.accuracy(HELDOUT, _have)
            print(f"{cand['name']}：沿用已评完的结果，正确率 {acc['overall']:.3f}")
        else:
            if _loaded:
                # vLLM does not reliably give the GPU back within one process: one candidate per run.
                print(f"\n{cand['name']} 还没评：请用“代码执行程序 → 重启会话”，再从头运行本 notebook（STAGE 仍是 \"veto\"）。"
                      "已评完的候选和已生成的批次都会沿用。")
                break
            local = Path("/content/veto_model")
            shutil.rmtree(local, ignore_errors=True)
            print(f"{cand['name']}：从 {cand['directory']} 复制到本地盘…", flush=True)
            shutil.copytree(cand["directory"], local)
            llm = cp.make_llm(local, MAX_MODEL_LEN, VLLM["gpu_memory_utilization"], VLLM["quantization"], VLLM["max_num_seqs"], SEED)
            _loaded = True
            results = run_samples(llm, HELDOUT, "heldout", SAMPLES_HELDOUT, _dir, "heldout")
            acc = cp.accuracy(HELDOUT, results)
            cp.atomic_json(out, {"name": cand["name"], "fingerprint": cand["fingerprint"], "eval_identity": EVAL_ID,
                                 "accuracy": acc, "versions": VERSIONS})
            print(f"{cand['name']}：正确率 {acc['overall']:.3f}（pi {SUBJECT_ACC['overall']:.3f}）", flush=True)
        EVALS.append((cand["name"], acc))
        DECISION = cp.veto(SUBJECT_ACC, EVALS, VETO_MAX_DROP)
        if DECISION["approved"]:
            break
    if DECISION is not None and (DECISION["approved"] or len(EVALS) == len(REQUEST["candidates"])):
        DECISION.update(request=REQUEST, versions=VERSIONS, eval_identity=EVAL_ID,
                        rule="通过条件：overall（各类正确率的等权平均）不比 pi 低超过 VETO_MAX_DROP；overall_micro、各类与配对差的区间只记录")
        cp.atomic_json(_req_path.parent / "result.json", DECISION)
        print(json.dumps({k: v for k, v in DECISION.items() if k != "request"}, ensure_ascii=False, indent=1))
        print("\n通过：" + (DECISION["approved"] or "没有候选通过；先不要上传，把上面的输出发回来"))


## 最后：自动断开

`AUTO_DISCONNECT = True` 时，这一格先把 Drive 写完，再断开运行时；结果都已经在 Drive 上。`False` 时什么也不做，记得手动“断开连接并删除运行时”。


In [ ]:
if AUTO_DISCONNECT:
    _release_runtime("全部跑完")
